# Getting Started

## Development of an Explainable Deep Learning Framework for Raman Spectroscopy-Based Bacterial Classification Using Three-Stage Transfer Learning

This notebook provides a step-by-step guide for reproducing the complete training pipeline of the project.

The notebook automatically detects whether it is running in Google Colab or a local Jupyter environment and configures the workflow accordingly.

When executed in Google Colab, experiment outputs can be stored in Google Drive for persistent storage. When executed locally, outputs are written to the repository's `experiments/` directory.

The scientific methodology remains identical across both execution environments.

The notebook supports both Google Colab and local Jupyter environments. When executed in Google Colab, experiment outputs can be stored directly in Google Drive for persistence. When executed locally, outputs are stored in the repository's experiment directory.

## Workflow Overview

The complete workflow consists of the following steps:

1. Detect the execution environment (Google Colab or local Jupyter)
2. Prepare the project workspace
3. Install project dependencies
4. Prepare the Raman spectroscopy datasets
5. Verify dataset availability
6. Train Stage 1: Reference Isolate Learning
7. Train Stage 2: Treatment Representation Learning
8. Train Stage 3: Clinical Transfer Learning
9. Review generated experiment artifacts

When executed in **Google Colab**, datasets and experiment outputs can be stored in Google Drive for persistence.

When executed **locally**, datasets are read from the repository's `data/raw/` directory and experiment outputs are written to the `experiments/` directory.

## Step 1. Configure the Execution Environment

This notebook automatically detects whether it is running in Google Colab or a local Jupyter environment.

For Google Colab, Google Drive is mounted automatically to enable persistent storage of datasets and experiment artifacts.

For local execution, the current repository directory is used and experiment outputs are stored locally.

In [7]:
from pathlib import Path
import os
import platform
import subprocess
import sys
import shutil

try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/raman-spectral-classifier")
    DATA_ROOT = PROJECT_ROOT / "data" / "raw"
    OUTPUT_ROOT = Path("/content/drive/MyDrive/raman_results")

    print("Google Colab environment detected.")

except ImportError:

    IN_COLAB = False

    current = Path.cwd().resolve()

    while current != current.parent:
        if (current / "requirements.txt").exists() and (current / "src").exists():
            PROJECT_ROOT = current
            break
        current = current.parent
    else:
        raise RuntimeError(
            "Could not locate the project root. "
            "Please run this notebook from inside the repository."
        )

    DATA_ROOT = PROJECT_ROOT / "data" / "raw"
    OUTPUT_ROOT = PROJECT_ROOT / "experiments"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    print("Local Jupyter environment detected.")

print(f"Project Root : {PROJECT_ROOT}")
print(f"Dataset Root : {DATA_ROOT}")
print(f"Output Root  : {OUTPUT_ROOT}")

OUTPUT_DIR = str(OUTPUT_ROOT)

Local Jupyter environment detected.
Project Root : S:\raman-spectral-classifier
Dataset Root : S:\raman-spectral-classifier\data\raw
Output Root  : S:\raman-spectral-classifier\experiments


## Step 2. Clone the Repository

Clone the latest version of the project repository and switch into the project directory.

In [8]:
if IN_COLAB:

    if not PROJECT_ROOT.exists():

        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/rana-rohit/raman-spectral-classifier.git",
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    os.chdir(PROJECT_ROOT)

else:

    print("Repository already available locally.")

    os.chdir(PROJECT_ROOT)

Repository already available locally.


In [9]:
print("=" * 60)
print("Repository Structure")
print("=" * 60)

for item in sorted(PROJECT_ROOT.iterdir()):
    print(item.name)

print(f"\nWorking directory: {Path.cwd()}")

Repository Structure
.git
.gitignore
artifacts
assets
CITATION.cff
configs
conftest.py
data
docs
experiments
LICENSE
metadata
notebooks
pyproject.toml
README.md
requirements.txt
scripts
src
tests
venv

Working directory: S:\raman-spectral-classifier


## Step 3. Install Project Dependencies

Install all required Python packages listed in `requirements.txt`. These include PyTorch, scientific computing libraries, visualization packages, and other project dependencies.

In [10]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(PROJECT_ROOT / "requirements.txt"),
    ],
    check=True,
)

CompletedProcess(args=['s:\\raman-spectral-classifier\\venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-r', 'S:\\raman-spectral-classifier\\requirements.txt'], returncode=0)

In [11]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Running on CPU.")

PyTorch : 2.11.0+cpu
CUDA Available : False
Running on CPU.


## Step 4. Prepare the Dataset

Prepare the Raman spectroscopy datasets.

When executed in Google Colab, the notebook copies the datasets from Google Drive into the repository.

When executed locally, the notebook verifies that the required dataset files are already present under `data/raw/`.


In [12]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:

    source = Path("/content/drive/MyDrive/Raman_spectral_classifier/data/raw")

    for file in source.glob("*.npy"):

        shutil.copy(file, DATA_ROOT)

    print("Dataset copied from Google Drive.")

else:

    print("Local execution detected.")

    print("Place the following files inside:")

    print(DATA_ROOT)

    print()

    required = [
        "X_reference.npy",
        "y_reference.npy",
        "X_finetune.npy",
        "y_finetune.npy",
        "X_2018clinical.npy",
        "X_2019clinical.npy",
        "wavenumbers.npy",
    ]

    for f in required:

        print("-", f)

Local execution detected.
Place the following files inside:
S:\raman-spectral-classifier\data\raw

- X_reference.npy
- y_reference.npy
- X_finetune.npy
- y_finetune.npy
- X_2018clinical.npy
- X_2019clinical.npy
- wavenumbers.npy


## Step 5. Verify Dataset Files

Verify that all required dataset files have been copied successfully before preprocessing begins.

In [13]:
required = [
    "X_reference.npy",
    "y_reference.npy",
    "X_finetune.npy",
    "y_finetune.npy",
    "X_2018clinical.npy",
    "X_2019clinical.npy",
    "wavenumbers.npy",
]

missing = []

for file in required:
    path = DATA_ROOT / file

    if path.exists():
        print(f"✓ {file}")
    else:
        print(f"✗ {file}")
        missing.append(file)

if missing:
    raise FileNotFoundError(
        "Missing dataset files:\n" + "\n".join(missing)
    )

✓ X_reference.npy
✓ y_reference.npy
✓ X_finetune.npy
✓ y_finetune.npy
✓ X_2018clinical.npy
✓ X_2019clinical.npy
✓ wavenumbers.npy


## Stage 1: Reference Isolate Learning

The first stage learns discriminative Raman spectral representations from the reference bacterial isolate dataset using 30-class classification.

The preprocessing pipeline is automatically executed before training.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/setup_data.py",
        "--stage",
        "s1_isolate",
    ],
    check=True,
)

### Train Stage 1 Model

This command trains the Temporal Convolutional Network (TCN) on the processed reference dataset.

The resulting checkpoint will later initialize Stage 2.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/train.py",
        "--model", "tcn",
        "--stage", "s1_isolate",
        "--override", "split_mode=iid_reference",
        "--exp-name", "tcn_stage1",
        "--exp-dir", OUTPUT_DIR,
    ],
    check=True,
)

## Stage 2: Treatment Representation Learning

Stage 2 transfers the learned isolate representations into treatment semantic space by remapping isolate labels into clinically relevant treatment groups.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/setup_data.py",
        "--stage",
        "s2_treatment",
    ],
    check=True,
)

### Train Stage 2 Model

The pretrained Stage 1 weights are further optimized using treatment-class supervision to learn clinically meaningful representations.

In [ ]:
 subprocess.run(
    [
        sys.executable,
        "scripts/train.py",
        "--model", "tcn",
        "--stage", "s2_treatment",
        "--override", "split_mode=iid_reference",
        "--exp-name", "tcn_stage2",
        "--exp-dir", OUTPUT_DIR,
    ],
    check=True,
)

In [ ]:
from pathlib import Path

STAGE2_CHECKPOINT = (
    OUTPUT_ROOT
    / "tcn_stage2"
    / "checkpoints"
    / "best_model.pt"
)

STAGE2_CHECKPOINT = str(STAGE2_CHECKPOINT)

print(STAGE2_CHECKPOINT)

## Stage 3: Clinical Transfer Learning

The final stage adapts the pretrained network to clinical Raman spectra using supervised fine-tuning and evaluates performance through patient-level cross-validation.

This stage initializes the model using the best checkpoint obtained from Stage 2.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/run_patient_cv.py",
        "--model", "tcn",
        "--stage", "s3_transfer",
        f"training.pretrained_checkpoint={STAGE2_CHECKPOINT}",
        "training.target_supervised.enabled=true",
        "--exp-name", "tcn_stage3",
        "--exp-dir", OUTPUT_DIR,
    ],
    check=True,
)

## Expected Outputs

After successful execution, the experiment directory will contain:

- Trained model checkpoints
- Training logs
- TensorBoard summaries (if enabled)
- Evaluation metrics
- Patient-level prediction results
- Explainability artifacts (generated separately)

The trained checkpoints can subsequently be used for model evaluation and explainability analysis.

## Reproducibility Notes

For reproducible experiments:

- Execute the notebook sequentially from top to bottom.
- Complete each training stage before proceeding to the next.
- Preserve the generated Stage 1 and Stage 2 checkpoints, as they are required for subsequent transfer learning stages.
- Use the same random seed and configuration files when reproducing the reported experimental results.

In [ ]:
## Troubleshooting

If you encounter issues while running this notebook:

- Ensure the required dataset files are available under `data/raw/`.
- Verify that all project dependencies have been installed successfully.
- When using Google Colab, connect to a GPU runtime before training.
- Complete Stage 1 before Stage 2, and Stage 2 before Stage 3.
- Verify that the Stage 2 checkpoint exists before starting clinical transfer learning.